In [ ]:
import os
import glob
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_bounds
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

# function for resampling a single raster
def adj_spatialres(input_raster, output_raster, res=0.5):
    """
    adjust raster to 0.5° x 0.5° -->  nearest-neighbor interpolation.
    """
    with rasterio.open(input_raster) as src:
        left, bottom, right, top = src.bounds
        width = int((right - left) / res)
        height = int((top - bottom) / res)
        new_transform = from_bounds(left, bottom, right, top, width, height)

        kwargs = src.meta.copy()
        kwargs.update({
            "height": height,
            "width": width,
            "transform": new_transform
        })

        dst = np.empty((height, width), dtype=src.meta['dtype'])

        reproject(
            source=rasterio.band(src, 1),
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=new_transform,
            dst_crs=src.crs,
            resampling=Resampling.nearest
        )

        with rasterio.open(output_raster, "w", **kwargs) as dst_file:
            dst_file.write(dst, 1)

# function to resample all rasters in a folder
def adj_all_spatialres(input_folder, output_folder, pattern="*.tif", res=0.5):
    """
    adjust all rasters in input_folder and save to output_folder.
    """
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    input_files = glob.glob(os.path.join(input_folder, pattern))

    for f in input_files:
        filename = os.path.basename(f)
        output_path = os.path.join(output_folder, filename.replace(".tif", f"_0.5deg.tif"))
        print(f"Resampling {filename} -> {os.path.basename(output_path)}")
        resample_to_half_degree(f, output_path, res=res)

# connect input & output files to drive
input_folder = "/content/drive/MyDrive/spatial-n2o-project/raw data"
output_folder = "/content/drive/MyDrive/spatial-n2o-project/processed data"

# run function
adj_all_spatialres(input_folder, output_folder)
